# Misc: Parsing Artikel Berita dengan `newspaper`

Untuk **artikel/berita**, ada library yang hampir ajaib: kasih URL → dapat **judul, isi
bersih, penulis, tanggal**, tanpa nulis selector satu per satu. Library ini memakai
heuristik untuk **menebak konten utama** dan membuang menu/iklan/footer (boilerplate removal).
Karena itu, **kode yang sama jalan di banyak situs berita** (multi-source).

> **Catatan instalasi:** `newspaper3k` (yang lawas) sering error di Python/lxml versi baru.
> Kita pakai penerusnya yang aktif dirawat: **`newspaper4k`** — API-nya **sama**
> (`from newspaper import Article`).

Install:

```bash
uv sync --extra news        # menambah newspaper4k
```

**Tooling:** `newspaper4k`. (Alternatif lain: `trafilatura`, `news-please`.)

> ⚠️ Ini **khusus artikel teks**. Untuk e-commerce (grid produk/harga) tetap pakai
> BeautifulSoup / XPath / API / Selenium.


## 1. Pemakaian dasar

Pola intinya **3 baris**:

```python
a = Article(url)
a.download()   # ambil HTML
a.parse()      # ekstrak judul, teks, penulis, tanggal
```

Di bawah kita coba pada satu URL nyata. Kalau jaringan/situs memblokir, kita pakai
**HTML contoh** lewat `set_html()` agar tetap bisa demo (deterministik).


In [1]:
from newspaper import Article

# HTML contoh (fallback) — sebuah artikel berita sederhana
SAMPLE_HTML = """
<html><head><title>Harga Kopi Naik 2026 - Berita Ekonomi</title></head>
<body>
  <nav>Beranda | Ekonomi | Olahraga | Teknologi</nav>
  <article>
    <h1>Harga Kopi Diproyeksikan Naik pada 2026</h1>
    <p class="byline">Oleh Andi Wijaya</p>
    <p>Harga kopi global diperkirakan naik tahun ini akibat cuaca ekstrem di sejumlah
       negara produsen utama.</p>
    <p>Petani di beberapa daerah melaporkan penurunan hasil panen yang cukup signifikan
       dibanding tahun lalu.</p>
    <p>Analis memperkirakan tren kenaikan ini berlanjut hingga akhir tahun.</p>
  </article>
  <footer>Hak cipta 2026 - Semua hak dilindungi.</footer>
</body></html>
"""


def parse_artikel(url, fallback_html=None):
    a = Article(url, language="id")
    try:
        a.download()
        a.parse()
        if not a.text and fallback_html:  # download sukses tapi teks kosong
            raise ValueError("teks kosong")
    except Exception as e:
        if fallback_html is None:
            raise
        print(f"(download gagal: {e} -> pakai HTML contoh)")
        a = Article(url, language="id")
        a.set_html(fallback_html)
        a.parse()
    return a


# coba URL nyata (Wikipedia: stabil & tidak memblokir); fallback ke HTML contoh
art = parse_artikel("https://en.wikipedia.org/wiki/Web_scraping", fallback_html=SAMPLE_HTML)

print("JUDUL   :", art.title)
print("PENULIS :", art.authors)
print("TANGGAL :", art.publish_date)
print("TEKS    :", art.text[:300].replace("\n", " "), "...")


JUDUL   : Web scraping
PENULIS : ['Contributors to Wikimedia projects']
TANGGAL : 2005-09-17 18:57:30+00:00
TEKS    : For broader coverage of this topic, see Data scraping.  Method of extracting data from websites  "Web scraper" redirects here. For websites that scrape content, see Scraper site.  Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.[1] Web scr ...


## 2. Multi-source — kode yang sama untuk banyak situs

Inilah kelebihannya: **selector tidak perlu diubah** per situs. Cukup ganti URL.
Di bawah kita loop beberapa URL; yang berhasil ditampilkan judul + panjang teksnya.

(Di dunia nyata ini bisa berisi URL dari situs berita yang berbeda-beda.)


In [2]:
urls = [
    "https://en.wikipedia.org/wiki/Web_scraping",
    "https://en.wikipedia.org/wiki/Data_engineering",
    "https://en.wikipedia.org/wiki/Beautiful_Soup_(HTML_parser)",
]

for url in urls:
    try:
        a = Article(url)
        a.download()
        a.parse()
        print(f"[OK]    {a.title[:55]:55} | {len(a.text):>6} karakter")
    except Exception as e:
        print(f"[GAGAL] {url} -> {e}")


[OK]    Web scraping                                            |  20386 karakter


[OK]    Data engineering                                        |   8170 karakter


[OK]    Beautiful Soup (HTML parser)                            |   2360 karakter


## 3. (Opsional) Fitur NLP: keyword & ringkasan

`newspaper` bisa menghasilkan **keyword** dan **ringkasan** lewat `article.nlp()`.
Fitur ini butuh `nltk` (data `punkt`). Install:

```bash
uv sync --extra news-nlp
```

Kalau `nltk` belum ada, sel berikut akan melewatinya dengan aman.

## Kapan pakai apa
- **Artikel/berita** (teks panjang) → `newspaper4k` / `trafilatura` (otomatis, multi-source).
- **E-commerce / tabel / data terstruktur** → BeautifulSoup, XPath, `pandas.read_html`, atau API.


In [3]:
# Fitur NLP butuh nltk; jalankan hanya jika tersedia
try:
    art.nlp()
    print("KEYWORDS :", art.keywords[:10])
    print("RINGKASAN:", art.summary[:300], "...")
except Exception as e:
    print("Lewati NLP (nltk belum terpasang / data belum diunduh):", e)
    print("Install dengan: uv sync --extra news-nlp")


[nltk_data] Downloading package punkt to /Users/seceng/nltk_data...


[nltk_data]   Unzipping tokenizers/punkt.zip.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/seceng/nltk_data...


KEYWORDS : ['web', 'scraping', 'to', 'in', 'that', 'is', 'for', 'or', 'from', 'be']
RINGKASAN: Web scraping, web harvesting, or web data extraction is data scraping used for extracting data from websites.
[1] Web scraping software may directly access the World Wide Web using the Hypertext Transfer Protocol or a web browser.
Contact scraping is a type of web scraping that is used as a componen ...


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
